<a href="https://colab.research.google.com/github/LuizFellipiFreire25/Projeto-ECAA08/blob/main/etapa-02-grafos/13%20-%20Algoritmos%20de%20Busca%20BFS%20e%20DFS%20em%20Tubulacoes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from collections import deque
from typing import List, Dict, Set, Optional

class GrafoNavegacaoAGV:
    """Estrutura do Grafo da Malha do AGV."""
    def __init__(self):
        self.adj: Dict[str, List[str]] = {}

    def adicionar_estacao(self, tag: str):
        if tag not in self.adj:
            self.adj[tag] = []

    def adicionar_corredor(self, origem: str, destino: str):
        if origem in self.adj and destino in self.adj:
            self.adj[origem].append(destino)


class BFS_AGV_Router:
    """Roteador BFS: Encontra o caminho com o menor número de conexões/hops."""
    def __init__(self, grafo: GrafoNavegacaoAGV):
        self.grafo = grafo

    def encontrar_rota_mais_curta(self, origem: str, destino: str, bloqueados: Optional[Set[str]] = None) -> Optional[List[str]]:
        bloqueados = bloqueados or set()
        if origem in bloqueados or destino in bloqueados:
            return None

        fila = deque([[origem]])
        visitados = {origem}

        while fila:
            caminho = fila.popleft()
            no_atual = caminho[-1]

            if no_atual == destino:
                return caminho

            for vizinho in self.grafo.adj.get(no_atual, []):
                if vizinho not in visitados and vizinho not in bloqueados:
                    visitados.add(vizinho)
                    nova_rota = list(caminho)
                    nova_rota.append(vizinho)
                    fila.append(nova_rota)

        return None


class DFS_AGV_PathFinder:
    """Buscador DFS: Mapeia todas as rotas de contingência possíveis até o destino."""
    def __init__(self, grafo: GrafoNavegacaoAGV):
        self.grafo = grafo

    def listar_todas_rotas(self, origem: str, destino: str, bloqueados: Optional[Set[str]] = None) -> List[List[str]]:
        bloqueados = bloqueados or set()
        rotas_encontradas = []

        def _dfs(no_atual: str, caminho_atual: List[str]):
            if no_atual == destino:
                rotas_encontradas.append(list(caminho_atual))
                return

            for vizinho in self.grafo.adj.get(no_atual, []):
                if vizinho not in caminho_atual and vizinho not in bloqueados:
                    caminho_atual.append(vizinho)
                    _dfs(vizinho, caminho_atual)
                    caminho_atual.pop()

        if origem not in bloqueados and destino not in bloqueados:
            _dfs(origem, [origem])

        return rotas_encontradas


# Execução do Teste de Roteamento
if __name__ == "__main__":
    # 1. Instanciação do Grafo
    malha = GrafoNavegacaoAGV()
    estacoes = ["ST-01", "DOC-101", "ALM-201", "AMO-301", "R-101", "DEP-401"]
    for e in estacoes:
        malha.adicionar_estacao(e)

    corredores = [
        ("ST-01", "DOC-101"),
        ("ST-01", "ALM-201"),
        ("DOC-101", "ALM-201"),
        ("ALM-201", "AMO-301"),
        ("AMO-301", "R-101"),
        ("AMO-301", "DEP-401"),
        ("R-101", "DEP-401"),
        ("DEP-401", "ST-01")
    ]
    for orig, dest in corredores:
        malha.adicionar_corredor(orig, dest)

    # 2. Teste de Busca BFS (Menor Rota em Hops)
    bfs = BFS_AGV_Router(malha)
    rota_rapida = bfs.encontrar_rota_mais_curta("ST-01", "DEP-401")
    print(f"BFS - Menor rota em waypoints (ST-01 -> DEP-401): {rota_rapida}")

    # 3. Teste de Contingência BFS com Ponto Bloqueado (Simulação de Obstáculo no AMO-301)
    obstaculos = {"AMO-301"}
    rota_com_obstaculo = bfs.encontrar_rota_mais_curta("ST-01", "DEP-401", bloqueados=obstaculos)
    print(f"BFS - Rota alternativa com bloqueio em AMO-301: {rota_com_obstaculo}")

    # 4. Teste de Busca DFS (Enumeração de Todas as Rotas de Contingência)
    dfs = DFS_AGV_PathFinder(malha)
    todas_rotas = dfs.listar_todas_rotas("ST-01", "DEP-401")
    print(f"\nDFS - Mapeamento completo de caminhos possíveis (ST-01 -> DEP-401):")
    for idx, rota in enumerate(todas_rotas, 1):
        print(f"  Opção {idx}: {' -> '.join(rota)}")

BFS - Menor rota em waypoints (ST-01 -> DEP-401): ['ST-01', 'ALM-201', 'AMO-301', 'DEP-401']
BFS - Rota alternativa com bloqueio em AMO-301: None

DFS - Mapeamento completo de caminhos possíveis (ST-01 -> DEP-401):
  Opção 1: ST-01 -> DOC-101 -> ALM-201 -> AMO-301 -> R-101 -> DEP-401
  Opção 2: ST-01 -> DOC-101 -> ALM-201 -> AMO-301 -> DEP-401
  Opção 3: ST-01 -> ALM-201 -> AMO-301 -> R-101 -> DEP-401
  Opção 4: ST-01 -> ALM-201 -> AMO-301 -> DEP-401
